# 01 - Gradient Practice

这本 notebook 用来手算梯度。建议每道题按这个顺序做：

1. 先只看题目，自己在纸上算 $\frac{\partial L}{\partial a}$ 和 $\frac{\partial L}{\partial b}$
2. 再运行代码 cell，用 micrograd 检查 `grad`
3. 最后点击 `Show / Hide 答案` 展开答案

记住规则：

- 加法：梯度原样传给两个输入
- 乘法：$out=xy$，则 $\frac{\partial out}{\partial x}=y$，$\frac{\partial out}{\partial y}=x$
- 平方：$out=x^2$，则 $\frac{\partial out}{\partial x}=2x$
- ReLU：输入大于 0 时导数是 1，小于 0 时导数是 0
- 一个变量有多条路径影响最终输出时，梯度要相加


<!-- xiao-preview:start -->
## 课前预习作业：两条局部反传规则

后面 10 道题都在反复用这两条规则。先填这两个最小例子，再开始手算题。


In [1]:
# TODO: 把 None 改成你手算出来的数字。
# 加法：out = a + b, 如果 out.grad = 7，a 和 b 各收到多少梯度？
preview_add_to_a = None
preview_add_to_b = None

# 乘法：out = a * b, a=2, b=3, 如果 out.grad = 5，a 和 b 各收到多少梯度？
preview_mul_to_a = None
preview_mul_to_b = None


def qa_check_01_preview(add_to_a, add_to_b, mul_to_a, mul_to_b):
    values = [add_to_a, add_to_b, mul_to_a, mul_to_b]
    if any(v is None for v in values):
        print('请先填写 TODO：先把加法和乘法的局部规则算出来。')
        return
    assert add_to_a == 7 and add_to_b == 7, '加法会把 out.grad 原样传给两边。'
    assert mul_to_a == 15, f'a 收到的梯度应是 out.grad * b = 5 * 3 = 15，但你填的是 {mul_to_a}'
    assert mul_to_b == 10, f'b 收到的梯度应是 out.grad * a = 5 * 2 = 10，但你填的是 {mul_to_b}'
    print('OK: 两条局部规则通过，开始做 10 道手算题。')


qa_check_01_preview(preview_add_to_a, preview_add_to_b, preview_mul_to_a, preview_mul_to_b)


请先填写 TODO：先把加法和乘法的局部规则算出来。


<!-- xiao-preview:hint -->
<details>
<summary><strong>Show / Hide 提示</strong></summary>

加法不改变梯度；乘法会乘上“另一个输入”。

</details>


<!-- xiao-preview:answer -->
<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
preview_add_to_a = 7
preview_add_to_b = 7
preview_mul_to_a = 15
preview_mul_to_b = 10
```

</details>


## 为什么要练手算

这一节只抓一句话：

> 先会手算小表达式的梯度，后面看 `backward()` 才不会像黑盒。

直觉上，micrograd 做的事并不神秘：

```text
每个小运算只知道自己的局部变化率；
反向传播把这些局部变化率沿路径乘起来；
如果同一个变量有多条路径影响 L，就把贡献加起来。
```

所以这本不是数学考试，而是在给源码理解铺路。

## 怎么练这本

这本不要一口气全跑完。正确方式是：

1. 先读题目，停下来手算
2. 写下你认为的 `a.grad` 和 `b.grad`
3. 再运行代码 cell 校验
4. 最后点击 `Show / Hide 答案` 展开答案

如果你直接一路 Run All，会失去练习效果。

第一轮只做 1-5 题；第二轮再做 6-10 题。


## 链式法则速记

遇到复杂一点的式子，先拆中间变量：

```text
x -> u -> y
```

就记：

$$
\frac{dy}{dx}=\frac{dy}{du}\cdot\frac{du}{dx}
$$

如果路径更长：

```text
x -> u -> v -> L
```

就继续乘：

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial v}
\cdot
\frac{\partial v}{\partial u}
\cdot
\frac{\partial u}{\partial x}
$$

如果同一个变量通过多条路径影响 $L$，就把每条路径的贡献相加。


## 手算模板

每题都按这个模板：

```text
1. 写出完整公式
2. 如果公式简单，直接求偏导
3. 如果公式有中间变量，画成几步：u=..., v=..., L=...
4. 使用链式法则：局部变化率相乘
5. 如果一个变量有多条路径影响 L，把路径贡献相加
```

小公式优先直接求偏导；路径法主要用来理解 micrograd 自动求导。


In [2]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / 'micrograd', cwd.parent / 'micrograd']
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'micrograd' / 'engine.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(f'Could not find local micrograd package from {cwd}')

sys.path.insert(0, str(PROJECT_ROOT))
from micrograd.engine import Value


def show_result(title, L, variables):
    print(title)
    print('L =', L)
    for name, value in variables.items():
        print(f'{name}.data = {value.data:>6}, {name}.grad = {value.grad:>8}')


def assert_close(actual, expected, name):
    assert abs(actual - expected) < 1e-9, f'{name}: expected {expected}, got {actual}'


def build_exercise(exercise_id):
    if exercise_id == 'ex1':
        a, b = Value(2.0), Value(3.0)
        L = a + b
    elif exercise_id == 'ex2':
        a, b = Value(2.0), Value(3.0)
        L = a * b
    elif exercise_id == 'ex3':
        a, b = Value(2.0), Value(3.0)
        L = a * b + a
    elif exercise_id == 'ex4':
        a, b = Value(2.0), Value(3.0)
        L = 2 * (a * b + a)
    elif exercise_id == 'ex5':
        a, b = Value(1.0), Value(2.0)
        L = (a + b) ** 2
    elif exercise_id == 'ex6':
        a, b = Value(2.0), Value(3.0)
        L = (a ** 2) * b + b
    elif exercise_id == 'ex7':
        a, b = Value(2.0), Value(3.0)
        L = (a * b + 1) ** 2
    elif exercise_id == 'ex8':
        a, b = Value(5.0), Value(2.0)
        L = (a - b) ** 2
    elif exercise_id == 'ex9':
        a, b = Value(2.0), Value(3.0)
        L = (a * b + a).relu()
    elif exercise_id == 'ex10':
        a, b = Value(2.0), Value(-3.0)
        L = (a * b + a).relu()
    else:
        raise ValueError(f'unknown exercise_id: {exercise_id}')
    return a, b, L


def expected_grads(exercise_id):
    a, b, L = build_exercise(exercise_id)
    L.backward()
    return a.grad, b.grad, L.data


def qa_check(exercise_id, your_a_grad, your_b_grad):
    if your_a_grad is None or your_b_grad is None:
        print(f'TODO: 请先填写 {exercise_id} 的 your_a_grad / your_b_grad，再运行本 cell。')
        return

    expected_a_grad, expected_b_grad, loss_data = expected_grads(exercise_id)
    assert_close(your_a_grad, expected_a_grad, f'{exercise_id} your_a_grad')
    assert_close(your_b_grad, expected_b_grad, f'{exercise_id} your_b_grad')
    print(f'OK: {exercise_id} passed. L.data = {loss_data}')


def qa_engine_self_test():
    for exercise_id in [f'ex{i}' for i in range(1, 11)]:
        expected_a_grad, expected_b_grad, _ = expected_grads(exercise_id)
        assert expected_a_grad is not None
        assert expected_b_grad is not None
    print('OK: qa_check engine is ready.')


qa_engine_self_test()

OK: qa_check engine is ready.


## 作业模式：先写你的答案

每道题下面都有一个留空 cell：

```python
your_a_grad = None
your_b_grad = None
qa_check('ex1', your_a_grad, your_b_grad)
```

做题时先把 `None` 改成你手算的答案，再运行 cell。

```text
如果还没填，qa_check 会提示 TODO。
如果填错，qa_check 会指出 expected / got。
如果填对，qa_check 会打印 OK。
```

提示和答案是分开的：先看 `Show / Hide 提示`，最后再看 `Show / Hide 答案`。

## Exercise 1 - Addition

题目：

$$
L = a + b
$$

取值：$a=2, b=3$

先手算：

$$
\frac{\partial L}{\partial a} = ?
$$

$$
\frac{\partial L}{\partial b} = ?
$$

In [3]:
# TODO: 把 None 改成你手算出来的数字。
your_a_grad = None
your_b_grad = None

qa_check('ex1', your_a_grad, your_b_grad)

TODO: 请先填写 ex1 的 your_a_grad / your_b_grad，再运行本 cell。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

加法的局部导数对两个输入都是 1。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

**答案**

$$\frac{\partial L}{\partial a}=1, \quad \frac{\partial L}{\partial b}=1$$

</details>

## Exercise 2 - Multiplication

题目：

$$
L = ab
$$

取值：$a=2, b=3$

先手算：

$$
\frac{\partial L}{\partial a} = ?
$$

$$
\frac{\partial L}{\partial b} = ?
$$

In [4]:
# TODO: 把 None 改成你手算出来的数字。
your_a_grad = None
your_b_grad = None

qa_check('ex2', your_a_grad, your_b_grad)

TODO: 请先填写 ex2 的 your_a_grad / your_b_grad，再运行本 cell。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

对 $a$ 求偏导时，把 $b$ 当常数；对 $b$ 求偏导时，把 $a$ 当常数。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

**答案**

$$\frac{\partial L}{\partial a}=b=3, \quad \frac{\partial L}{\partial b}=a=2$$

</details>

## Exercise 3 - One Variable Has Two Paths

题目：

$$
L = ab + a
$$

取值：$a=2, b=3$

先手算：

$$
\frac{\partial L}{\partial a} = ?
$$

$$
\frac{\partial L}{\partial b} = ?
$$

In [5]:
# TODO: 把 None 改成你手算出来的数字。
your_a_grad = None
your_b_grad = None

qa_check('ex3', your_a_grad, your_b_grad)

TODO: 请先填写 ex3 的 your_a_grad / your_b_grad，再运行本 cell。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

$a$ 有两条路径影响 $L$：一条经过 $ab$，一条是直接的 $+a$。两条路径贡献要相加。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

**答案**

$$\frac{\partial L}{\partial a}=b+1=4, \quad \frac{\partial L}{\partial b}=a=2$$

</details>

## Exercise 4 - The micrograd Example

题目：

$$
L = 2(ab + a)
$$

取值：$a=2, b=3$

先手算：

$$
\frac{\partial L}{\partial a} = ?
$$

$$
\frac{\partial L}{\partial b} = ?
$$

In [6]:
# TODO: 把 None 改成你手算出来的数字。
your_a_grad = None
your_b_grad = None

qa_check('ex4', your_a_grad, your_b_grad)

TODO: 请先填写 ex4 的 your_a_grad / your_b_grad，再运行本 cell。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

先算括号 $ab+a$ 的梯度，再整体乘上外面的 2。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

**答案**

$$\frac{\partial L}{\partial a}=2(b+1)=8, \quad \frac{\partial L}{\partial b}=2a=4$$

</details>

## Exercise 5 - Square After Addition

题目：

$$
L = (a+b)^2
$$

取值：$a=1, b=2$

先手算：

$$
\frac{\partial L}{\partial a} = ?
$$

$$
\frac{\partial L}{\partial b} = ?
$$

In [7]:
# TODO: 把 None 改成你手算出来的数字。
your_a_grad = None
your_b_grad = None

qa_check('ex5', your_a_grad, your_b_grad)

TODO: 请先填写 ex5 的 your_a_grad / your_b_grad，再运行本 cell。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

令 $u=a+b$，先算 $L=u^2$，再乘上 $u$ 对 $a,b$ 的局部导数。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

**答案**

令 $u=a+b=3$，$L=u^2$。所以：$$\frac{\partial L}{\partial a}=2u=6, \quad \frac{\partial L}{\partial b}=2u=6$$

</details>

## Exercise 6 - Repeated Variable In Multiplication

题目：

$$
L = a^2b + b
$$

取值：$a=2, b=3$

先手算：

$$
\frac{\partial L}{\partial a} = ?
$$

$$
\frac{\partial L}{\partial b} = ?
$$

In [8]:
# TODO: 把 None 改成你手算出来的数字。
your_a_grad = None
your_b_grad = None

qa_check('ex6', your_a_grad, your_b_grad)

TODO: 请先填写 ex6 的 your_a_grad / your_b_grad，再运行本 cell。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

$a^2b$ 对 $a$ 求导会出现 $2a$；对 $b$ 求导时，$a^2$ 当成常数，最后还要加上 $+b$ 的贡献。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

**答案**

$$\frac{\partial L}{\partial a}=2ab=12, \quad \frac{\partial L}{\partial b}=a^2+1=5$$

</details>

## Exercise 7 - Chain Rule Twice

题目：

$$
L = (ab + 1)^2
$$

取值：$a=2, b=3$

先手算：

$$
\frac{\partial L}{\partial a} = ?
$$

$$
\frac{\partial L}{\partial b} = ?
$$

In [9]:
# TODO: 把 None 改成你手算出来的数字。
your_a_grad = None
your_b_grad = None

qa_check('ex7', your_a_grad, your_b_grad)

TODO: 请先填写 ex7 的 your_a_grad / your_b_grad，再运行本 cell。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

令 $u=ab+1$。先算平方的导数 $2u$，再乘 $u$ 对 $a,b$ 的偏导。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

**答案**

令 $u=ab+1=7$，$L=u^2$。所以：$$\frac{\partial L}{\partial a}=2u\cdot b=42, \quad \frac{\partial L}{\partial b}=2u\cdot a=28$$

</details>

## Exercise 8 - Negative Values

题目：

$$
L = (a-b)^2
$$

取值：$a=5, b=2$

先手算：

$$
\frac{\partial L}{\partial a} = ?
$$

$$
\frac{\partial L}{\partial b} = ?
$$

In [10]:
# TODO: 把 None 改成你手算出来的数字。
your_a_grad = None
your_b_grad = None

qa_check('ex8', your_a_grad, your_b_grad)

TODO: 请先填写 ex8 的 your_a_grad / your_b_grad，再运行本 cell。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

令 $u=a-b$。注意 $u$ 对 $b$ 的导数是 -1，所以 $b.grad$ 会带负号。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

**答案**

令 $u=a-b=3$，$L=u^2$。所以：$$\frac{\partial L}{\partial a}=2u=6, \quad \frac{\partial L}{\partial b}=-2u=-6$$

</details>

## Exercise 9 - ReLU Positive Side

题目：

$$
L = \operatorname{ReLU}(ab + a)
$$

取值：$a=2, b=3$

先手算：

$$
\frac{\partial L}{\partial a} = ?
$$

$$
\frac{\partial L}{\partial b} = ?
$$

In [11]:
# TODO: 把 None 改成你手算出来的数字。
your_a_grad = None
your_b_grad = None

qa_check('ex9', your_a_grad, your_b_grad)

TODO: 请先填写 ex9 的 your_a_grad / your_b_grad，再运行本 cell。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

先算 ReLU 里面的值。如果它大于 0，ReLU 的局部导数是 1。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

**答案**

内部值 $ab+a=8>0$，ReLU 的局部导数是 1。 所以：$$\frac{\partial L}{\partial a}=b+1=4, \quad \frac{\partial L}{\partial b}=a=2$$

</details>

## Exercise 10 - ReLU Negative Side

题目：

$$
L = \operatorname{ReLU}(ab + a)
$$

取值：$a=2, b=-3$

先手算：

$$
\frac{\partial L}{\partial a} = ?
$$

$$
\frac{\partial L}{\partial b} = ?
$$

In [12]:
# TODO: 把 None 改成你手算出来的数字。
your_a_grad = None
your_b_grad = None

qa_check('ex10', your_a_grad, your_b_grad)

TODO: 请先填写 ex10 的 your_a_grad / your_b_grad，再运行本 cell。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

先算 ReLU 里面的值。如果它小于 0，ReLU 输出 0，局部导数是 0。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

**答案**

内部值 $ab+a=-4<0$，ReLU 输出 0，局部导数是 0。 所以：$$\frac{\partial L}{\partial a}=0, \quad \frac{\partial L}{\partial b}=0$$

</details>

## QA Engine Check

这格只检查 `qa_check` 这套测试工具可用，不展示每题答案。

In [13]:
qa_engine_self_test()

OK: qa_check engine is ready.


## Suggested Pace

第一轮只做 1 到 5 题，重点练熟加法、乘法和链式法则。

第二轮再做 6 到 10 题，重点看重复变量、多层链式法则、负号和 ReLU。

如果你某题算错，不要急着看答案。先打开提示，再画计算图，最后才看答案。

<!-- xiao-post:start -->
## 课后作业提交清单

这一节学完后，用下面 4 条自检：

```text
1. 我至少独立完成了前 5 题，再看提示或答案。
2. 我能解释加法、乘法、平方、ReLU 的局部反传规则。
3. 我能说清楚重复变量为什么要把梯度相加。
4. 我能把任意一道错题重新画成计算图。
```


## AI 复盘检查 Prompt

把下面这段发送给任意 AI，让它检查你是否真的能手算梯度：

```text
你是我的 micrograd 梯度手算练习检查官。

我刚学完一节内容，主题是：对简单标量表达式手算 dL/da 和 dL/db，并用 micrograd 的 grad 做校验。

请你用一问一答的方式检查我。规则：
1. 一次只出一道题，等我回答后再评价。
2. 不要一开始直接给答案。
3. 如果我答错了，不要立刻给完整答案，先提示我应该把哪个变量当常数，或者应该先设哪个中间变量。
4. 每道题要求我写出 dL/da 和 dL/db。
5. 最后给我一个 0-100 的掌握度评分，并指出我最薄弱的是加法、乘法、平方、链式法则、还是 ReLU。

请按这个顺序出题检查我：
1. L = a + b，a=2，b=3
2. L = ab，a=2，b=3
3. L = ab + a，a=2，b=3
4. L = 2(ab+a)，a=2，b=3
5. L = (a+b)^2，a=1，b=2
6. L = (ab+1)^2，a=2，b=3
7. L = ReLU(ab+a)，分别检查 ab+a > 0 和 ab+a < 0 两种情况

现在从第 1 题开始问我。
```
